# Template 00: EDA for All Features

**Purpose:** Analyze all columns and generate encoding recommendations

**Outputs:**
- `config_generated/auto_feature_encoding.csv` - Auto-detected encoding recommendations
- `config_generated/auto_dtype_fixes.csv` - Auto-detected dtype conversions
- `config_generated/master_feature_encoding.csv` - Merged manual + auto (used by pipeline)
- `config_generated/master_dtype_fixes.csv` - Merged manual + auto (used by pipeline)
- Visualizations (histograms and bar charts)

**Note:** Run this manually (not part of automated pipeline)

In [ ]:
# Parameters (can be overridden)
config_path = "config/car_coll/v1"
sample_size = 30000  # Number of rows to sample for analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
import os
import sys
from pathlib import Path

# Find project root (works from templates/ or output/model/v1/notebooks/)
current_dir = Path.cwd()
if current_dir.name == 'templates':
    project_root = current_dir.parent
elif current_dir.name == 'notebooks':
    # From output/model/v1/notebooks -> go up 4 levels to project root
    project_root = current_dir.parent.parent.parent.parent
else:
    project_root = current_dir

# Change to project root so all relative paths work
os.chdir(project_root)

# Add lib to path
lib_path = str(project_root / 'lib')
if lib_path not in sys.path:
    sys.path.insert(0, lib_path)

from utils import setup_notebook_environment, load_config

print("="*70)
print("EDA FOR ALL FEATURES")
print("="*70)
print(f"Project root: {project_root}")
print(f"Working dir: {os.getcwd()}")
print(f"Config path: {config_path}")
print(f"Sample size: {sample_size:,} rows")

In [ ]:
# Load config
config_file = f"{config_path}/config.yaml"
cfg = load_config(config_file)

# Setup paths
config_gen_dir = f"{config_path}/config_generated"
os.makedirs(config_gen_dir, exist_ok=True)

print(f"\n✓ Config loaded: {cfg['experiment']['name']}")
print(f"✓ Generated config dir: {config_gen_dir}")

## Step 1: Load Column Metadata

In [ ]:
# Load all_columns_master.csv
master_cols_file = f"{config_path}/all_columns_master.csv"
all_cols_df = pd.read_csv(master_cols_file)

print(f"\n✓ Loaded {len(all_cols_df)} columns from all_columns_master.csv")
print(f"\nColumn types:")
print(all_cols_df['dtype'].value_counts())

all_cols_df.head()

## Step 2: Load Sample Data

In [ ]:
# Load sample data
data_root = cfg['paths']['master_data_path']
master_file = f"{data_root}/{cfg['data']['master_file']}"

print(f"\n📂 Loading sample data from: {master_file}")
print(f"   Sample size: {sample_size:,} rows")

# Read parquet with row limit
df_sample = pd.read_parquet(master_file).head(sample_size)

print(f"\n✓ Loaded sample: {df_sample.shape}")
print(f"  Memory: {df_sample.memory_usage(deep=True).sum() / 1e6:.1f} MB")

## Step 3: Analyze Columns

In [ ]:
from eda_helpers import analyze_column, detect_dtype_conversions, merge_encodings

results = []
target_exposure = {cfg['experiment']['target'], cfg['experiment']['exposure']}

for _, row in all_cols_df.iterrows():
    col = row['column_name']
    if col in df_sample.columns:
        results.append(analyze_column(col, df_sample[col], target_exposure=target_exposure))

auto_df = pd.DataFrame(results)
print(f'Analyzed {len(auto_df)} columns')

## Step 3b: Visualizations

In [ ]:
# Encoding distribution
print('Encoding Distribution:')
encoding_counts = auto_df['encoding'].value_counts()
print(encoding_counts)
print(f'\nTotal columns analyzed: {len(auto_df)}')

# Bar chart
plt.figure(figsize=(10, 6))
encoding_counts.plot(kind='barh')
plt.title('Feature Encoding Recommendations')
plt.xlabel('Count')
plt.ylabel('Encoding Type')
plt.tight_layout()
plt.show()

In [ ]:
# Category distribution
print('\nCategory Distribution:')
category_counts = auto_df['category'].value_counts()
print(category_counts)

# Bar chart
plt.figure(figsize=(12, 6))
category_counts.plot(kind='barh')
plt.title('Feature Category Distribution')
plt.xlabel('Count')
plt.ylabel('Category')
plt.tight_layout()
plt.show()

In [ ]:
# Null percentage distribution
plt.figure(figsize=(10, 6))
auto_df['null_pct'].hist(bins=50)
plt.title('Null Percentage Distribution')
plt.xlabel('Null %')
plt.ylabel('Number of Features')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'\nFeatures with >50% nulls: {(auto_df["null_pct"] > 50).sum()}')
print(f'Features with >90% nulls: {(auto_df["null_pct"] > 90).sum()}')

In [ ]:
# Unique value distribution (log scale)
plt.figure(figsize=(10, 6))
auto_df['unique_count'].hist(bins=50, log=True)
plt.title('Unique Value Count Distribution (log scale)')
plt.xlabel('Unique Values')
plt.ylabel('Number of Features (log)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'\nHigh cardinality features (>100 unique): {(auto_df["unique_count"] > 100).sum()}')

## Step 3c: Individual Feature Distributions

Scroll through all features to see their distributions.

In [ ]:
# Show distribution for each feature
import warnings
warnings.filterwarnings('ignore')

for idx, row in auto_df.iterrows():
    feature_num = idx + 1
    col_name = row['column_name']
    encoding = row['encoding']
    category = row['category']
    null_pct = row['null_pct']
    
    # Skip if column not in sample
    if col_name not in df_sample.columns:
        continue
    
    # Skip identifiers and skipped features (too many to show)
    if encoding == 'skip':
        continue
    
    col_data = df_sample[col_name]
    
    # Create figure
    plt.figure(figsize=(12, 4))
    
    # Title with feature info
    title = f"#{feature_num}: {col_name}\n"
    title += f"Encoding: {encoding} | Category: {category} | Nulls: {null_pct}%"
    
    # Numeric features - histogram
    if encoding in ['numeric', 'ordinal_0_5']:
        plt.subplot(1, 2, 1)
        col_data.dropna().hist(bins=50, edgecolor='black')
        plt.title(title)
        plt.xlabel('Value')
        plt.ylabel('Frequency')
        plt.grid(alpha=0.3)
        
        # Stats
        plt.subplot(1, 2, 2)
        stats_text = f"Count: {col_data.count():,}\n"
        stats_text += f"Mean: {col_data.mean():.2f}\n"
        stats_text += f"Std: {col_data.std():.2f}\n"
        stats_text += f"Min: {col_data.min():.2f}\n"
        stats_text += f"25%: {col_data.quantile(0.25):.2f}\n"
        stats_text += f"50%: {col_data.median():.2f}\n"
        stats_text += f"75%: {col_data.quantile(0.75):.2f}\n"
        stats_text += f"Max: {col_data.max():.2f}"
        plt.text(0.1, 0.5, stats_text, fontsize=12, family='monospace',
                verticalalignment='center')
        plt.axis('off')
    
    # Categorical features - value counts
    else:
        value_counts = col_data.value_counts()
        
        # Show top 20 values if high cardinality
        if len(value_counts) > 20:
            value_counts = value_counts.head(20)
            note = f" (showing top 20 of {col_data.nunique()} unique values)"
        else:
            note = ""
        
        plt.subplot(1, 1, 1)
        value_counts.plot(kind='barh')
        plt.title(title + note)
        plt.xlabel('Count')
        plt.ylabel('Value')
        plt.grid(alpha=0.3, axis='x')
    
    plt.tight_layout()
    plt.show()
    
    # Add small separator
    print('-' * 80)

print(f'\nShown distributions for non-skipped features')

## Step 4: Save & Merge

In [ ]:
auto_df.to_csv(f'{config_gen_dir}/auto_feature_encoding.csv', index=False)
print(f'Saved auto_feature_encoding.csv')

dtype_df = detect_dtype_conversions(df_sample)
if len(dtype_df) > 0:
    dtype_df.to_csv(f'{config_gen_dir}/auto_dtype_fixes.csv', index=False)

manual_file = f'{config_path}/manual_feature_encoding.csv'
master_df = merge_encodings(auto_df, manual_file)
master_df.to_csv(f'{config_gen_dir}/master_feature_encoding.csv', index=False)
print(f'Saved master_feature_encoding.csv')

master_df.head(20)

## Summary

In [ ]:
print(f'Total: {len(master_df)}')
print(f"Manual: {(master_df['source']=='manual').sum()}")
print(f"Auto: {(master_df['source']=='auto').sum()}")
print(f"\nEncoding types:")
print(master_df['encoding'].value_counts())